# Recap całego semestru

Ten notebook podsumowuje wszystko co przerobiliśmy przez semestr:

1. **OOP** — klasy, dziedziczenie, dataclasses, ABC
2. **Pydantic** — walidacja danych
3. **NLP** — przetwarzanie tekstu (spaCy, transformery)
4. **Machine Learning** — regresja, drzewa, random forest, ewaluacja

In [1]:
# Importy wspólne dla całego notebooka
import numpy as np
import pandas as pd

# Część 1 — Programowanie obiektowe (OOP)

Fundament całego kursu. Klasy pozwalają grupować dane i zachowania w jeden obiekt. Dziedziczenie pozwala budować hierarchie i unikać powtarzania kodu.

## 1.1 Klasy, dziedziczenie, nadpisywanie metod

**Klasa** to szablon obiektu. **Dziedziczenie** pozwala stworzyć nową klasę na bazie istniejącej. **Nadpisywanie** to zastąpienie metody rodzica własną wersją.

In [2]:
class Animal:
    def __init__(self, name, age):
        # __init__ to konstruktor — wywoływany przy tworzeniu obiektu
        # self to odniesienie do konkretnego obiektu
        self.name = name      # atrybut instancji
        self.age = age

    def speak(self):
        # metoda bazowa — podklasy ją nadpiszą
        return '...'

    def describe(self):
        # metoda korzysta z speak() — która może być nadpisana
        return f'{self.name} ({self.age} lat): {self.speak()}'


class Dog(Animal):
    # Dog dziedziczy po Animal — ma name, age, describe()
    def speak(self):
        # nadpisanie metody speak() z klasy Animal
        return 'Hau!'


class Cat(Animal):
    def __init__(self, name, age, indoor=True):
        # super() wywołuje __init__ rodzica (Animal)
        # dzięki temu nie powtarzamy przypisania name i age
        super().__init__(name, age)
        self.indoor = indoor   # dodatkowy atrybut tylko dla Cat

    def speak(self):
        return 'Miau!'


# Tworzymy obiekty
rex = Dog('Rex', 3)
whiskers = Cat('Wąsik', 5, indoor=True)

print(rex.describe())
print(whiskers.describe())

Rex (3 lat): Hau!
Wąsik (5 lat): Miau!


### Polimorfizm

Polimorfizm = ta sama operacja (`speak()`) zachowuje się różnie zależnie od typu obiektu. Pętla nie musi wiedzieć czy ma psa czy kota — po prostu woła `describe()`.

In [3]:
# Lista różnych zwierząt — wspólny interfejs
animals = [Dog('Rex', 3), Cat('Wąsik', 5), Animal('Nieznane', 1)]

for a in animals:
    # ta sama metoda, różne zachowanie — to jest polimorfizm
    print(f'{a.__class__.__name__:8} → {a.speak()}')

Dog      → Hau!
Cat      → Miau!
Animal   → ...


## 1.2 Dataclasses

`@dataclass` automatycznie generuje `__init__`, `__repr__` i `__eq__` na podstawie adnotacji typów. Mniej kodu, mniej błędów.

In [4]:
from dataclasses import dataclass, field


@dataclass
class Product:
    name: str                                  # pole wymagane
    price: float
    in_stock: bool = True                      # pole z wartością domyślną
    tags: list = field(default_factory=list)   # mutable default — NIGDY tags: list = []


p1 = Product('Laptop', 3999.99)
p2 = Product('Laptop', 3999.99)

print(p1)              # __repr__ wygenerowany automatycznie
print(p1 == p2)        # __eq__ wygenerowany — porównuje pola, nie adresy w pamięci

# Dlaczego default_factory? Gdyby bwyło tags=[], wszystkie obiekty
# współdzieliłyby JEDNĄ listę — klasyczny błąd w Pythonie.
p1.tags.append('promocja')
print(f'p1.tags: {p1.tags}')   # ['promocja']
print(f'p2.tags: {p2.tags}')   # [] — osobna lista, dobrze!

Product(name='Laptop', price=3999.99, in_stock=True, tags=[])
True
p1.tags: ['promocja']
p2.tags: []


## 1.4 ABC — klasy abstrakcyjne -- nie obowiazuje na egzaminie

ABC (Abstract Base Class) wymusza żeby podklasy implementowały określone metody. Próba stworzenia obiektu bez implementacji — błąd już przy tworzeniu, nie przy wywołaniu.

**To kluczowy wzorzec** — zobaczycie go ponownie w sklearn, gdzie każdy model musi mieć `fit()` i `predict()`.

In [5]:
from abc import ABC, abstractmethod


class Shape(ABC):
    @abstractmethod
    def area(self):
        # metoda abstrakcyjna — podklasa MUSI ją zaimplementować
        pass

    def describe(self):
        # metoda konkretna — wspólna dla wszystkich kształtów
        return f'{self.__class__.__name__}: pole = {self.area():.2f}'


class Circle(Shape):
    def __init__(self, r):
        self.r = r
    def area(self):                  # implementacja wymaganej metody
        return 3.14159 * self.r ** 2


class Square(Shape):
    def __init__(self, a):
        self.a = a
    def area(self):
        return self.a ** 2


# Polimorfizm — wspólny interfejs, różne implementacje
for shape in [Circle(5), Square(4)]:
    print(shape.describe())

Circle: pole = 78.54
Square: pole = 16.00


# Część 2 — Pydantic

Pydantic to biblioteka do walidacji danych. Tam gdzie dataclass tylko przechowuje dane, Pydantic **sprawdza ich poprawność** i **konwertuje typy**. Niezbędne gdy dane przychodzą z zewnątrz — API, formularzy, plików.

## 2.1 Podstawowa walidacja

Model Pydantic wygląda jak dataclass, ale automatycznie waliduje typy i wartości.

In [6]:
from pydantic import BaseModel, Field, field_validator
from typing import Optional


class User(BaseModel):
    name: str
    age: int = Field(ge=0, le=120)    # ge=greater-equal, le=less-equal
    email: str

    @field_validator('email')
    @classmethod
    def email_valid(cls, v):
        # własna logika walidacji
        if '@' not in v:
            raise ValueError('niepoprawny email')
        return v.lower()             # normalizacja — zwracamy zmodyfikowaną wartość


# Pydantic AUTOMATYCZNIE konwertuje '25' (string) na 25 (int)
u = User(name='Anna', age='25', email='ANNA@EXAMPLE.COM')
print(f'age: {u.age}, typ: {type(u.age).__name__}')   # 25, int
print(f'email: {u.email}')                            # znormalizowany do małych liter

age: 25, typ: int
email: anna@example.com


In [7]:
# Walidacja w akcji — błędne dane dają czytelny błąd
try:
    bad = User(name='Jan', age=150, email='jan@example.com')
except Exception as e:
    print('Walidacja zadziałała — age=150 przekracza limit 120')

try:
    bad2 = User(name='Ola', age=30, email='zly-email')
except Exception as e:
    print('Walidacja zadziałała — email bez @ odrzucony')

Walidacja zadziałała — age=150 przekracza limit 120
Walidacja zadziałała — email bez @ odrzucony


## 2.2 Zagnieżdżone modele i JSON -- nie obowiazuje na egzaminie

Pydantic obsługuje modele zagnieżdżone i automatyczne parsowanie słowników/JSON. To dlatego jest standardem w API (np. FastAPI).

In [8]:
class Address(BaseModel):
    city: str
    zip_code: str


class Person(BaseModel):
    name: str
    address: Optional[Address] = None    # zagnieżdżony model, opcjonalny


# Dane jako słownik (np. z API albo pliku JSON)
data = {
    'name': 'Anna Kowalska',
    'address': {'city': 'Warszawa', 'zip_code': '00-001'}
}

# Pydantic parsuje zagnieżdżoną strukturę automatycznie
person = Person(**data)
print(person.address.city)         # Warszawa — address to obiekt Address
print()
print('Eksport z powrotem do JSON:')
print(person.model_dump_json(indent=2))

Warszawa

Eksport z powrotem do JSON:
{
  "name": "Anna Kowalska",
  "address": {
    "city": "Warszawa",
    "zip_code": "00-001"
  }
}


# Część 3 — Przetwarzanie języka naturalnego (NLP)

Jak zamienić tekst na coś co komputer może przetwarzać. Od najprostszego zliczania słów, przez klasyczne narzędzia (spaCy), po nowoczesne modele (transformery).

## 3.1 Tekst jako dane — bag of words -- nie obowiazuje na egzaminie

Najprostsze NLP: liczymy ile razy występuje każde słowo. Komputer nie rozumie słów — rozumie liczby.

In [9]:
from collections import Counter

text = 'pies kot pies ptak pies kot'

# Counter zlicza wystąpienia każdego elementu
counts = Counter(text.split())
print(counts)                  # {'pies': 3, 'kot': 2, 'ptak': 1}
print(counts.most_common(2))   # 2 najczęstsze: [('pies', 3), ('kot', 2)]

# To jest 'bag of words' — reprezentacja tekstu jako liczby słów.
# Wada: traci kolejność. 'pies gryzie kota' i 'kot gryzie psa'
# dają prawie identyczny wynik. Ten problem rozwiązują transformery.

Counter({'pies': 3, 'kot': 2, 'ptak': 1})
[('pies', 3), ('kot', 2)]


## 3.2 spaCy — klasyczne NLP

spaCy to biblioteka do profesjonalnego przetwarzania tekstu: tokenizacja, części mowy (POS), rozpoznawanie nazw własnych (NER). Szybka, działa offline.

*Instalacja modelu polskiego zajmuje chwilę przy pierwszym uruchomieniu.*

In [10]:
# Instalacja spaCy i polskiego modelu
!pip install spacy -q
!python -m spacy download pl_core_news_sm -q
print('spaCy gotowe!')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 15.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pl_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
spaCy gotowe!


In [11]:
import spacy

# ładujemy polski model
nlp = spacy.load('pl_core_news_sm')

doc = nlp('Anna Kowalska pracuje w Google w Warszawie.')

# Tokenizacja + części mowy + lematyzacja
print('TOKENY:')
for token in doc[:5]:
    # token.text   = oryginalne słowo
    # token.pos_   = część mowy (PROPN=nazwa własna, VERB=czasownik...)
    # token.lemma_ = forma podstawowa (pracuje -> pracować)
    print(f'  {token.text:12} {token.pos_:8} lemat: {token.lemma_}')

print('\nROZPOZNANE JEDNOSTKI (NER):')
for ent in doc.ents:
    # model wykrywa osoby, miejsca, organizacje
    print(f'  {ent.text:18} → {ent.label_}')

TOKENY:
  Anna         PROPN    lemat: Anna
  Kowalska     PROPN    lemat: Kowalska
  pracuje      VERB     lemat: pracować
  w            ADP      lemat: w
  Google       PROPN    lemat: Google

ROZPOZNANE JEDNOSTKI (NER):
  Anna Kowalska      → persName
  Google             → orgName
  Warszawie          → placeName


## 3.3 Transformery — Hugging Face

Transformery to obecny standard w NLP. W przeciwieństwie do bag of words rozumieją kontekst — to samo słowo w różnych zdaniach jest interpretowane różnie.

`pipeline` to najprostszy sposób użycia gotowego modelu — 3 linijki kodu.

*Pierwszy model pobiera się ~500MB, chwilę to trwa.*

In [12]:
# Instalacja transformers (kliknij, poczekaj)
!pip install transformers -q
print('transformers gotowe!')

transformers gotowe!


In [13]:
from transformers import pipeline

# pipeline pobiera gotowy, wytrenowany model
# ten model robi analizę sentymentu i działa wielojęzycznie (też po polsku)
sentiment = pipeline(
    'text-classification',
    model='cardiffnlp/twitter-xlm-roberta-base-sentiment'
)

texts = [
    'Ten produkt jest absolutnie świetny, polecam wszystkim!',
    'Okropna obsługa klienta, nigdy więcej.',
    'Normalny produkt, nic szczególnego.',
]

for t in texts:
    r = sentiment(t)[0]
    # label = etykieta (positive/negative/neutral)
    # score = pewność modelu (0-1)
    print(f"{r['label']:10} ({r['score']:.2f}) → {t}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/841 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

positive   (0.94) → Ten produkt jest absolutnie świetny, polecam wszystkim!
negative   (0.88) → Okropna obsługa klienta, nigdy więcej.
positive   (0.45) → Normalny produkt, nic szczególnego.


# Część 4 — Machine Learning

Sedno drugiej połowy kursu. Modele które uczą się z danych zamiast być zaprogramowane ręcznie. Wszystkie używają tego samego interfejsu sklearn: `fit()` i `predict()` — dokładnie tak jak ABC z części 1.

## 4.1 Przygotowanie danych

Dwa klasyczne datasety:
- **Breast Cancer** — klasyfikacja (złośliwy/łagodny guz)
- **California Housing** — regresja (cena mieszkania)

Zawsze dzielimy dane na **train** (trening) i **test** (ocena). Model oceniamy na danych których nie widział.

In [14]:
from sklearn.datasets import load_breast_cancer, fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# --- DANE DO KLASYFIKACJI ---
cancer = load_breast_cancer()
Xc = pd.DataFrame(cancer.data, columns=cancer.feature_names)   # cechy (X)
yc = cancer.target                                            # etykiety (y): 0/1

# podział 80% trening / 20% test
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(
    Xc, yc, test_size=0.2, random_state=42
)

# skalowanie — sprowadza cechy do podobnej skali (potrzebne dla regresji logistycznej)
scaler = StandardScaler()
Xc_tr_s = scaler.fit_transform(Xc_tr)   # fit+transform na treningu
Xc_te_s = scaler.transform(Xc_te)        # tylko transform na teście (bez data leakage!)

# --- DANE DO REGRESJI ---
housing = fetch_california_housing()
Xh = pd.DataFrame(housing.data, columns=housing.feature_names)
yh = housing.target                                          # cena (liczba ciągła)
Xh_tr, Xh_te, yh_tr, yh_te = train_test_split(
    Xh, yh, test_size=0.2, random_state=42
)

print(f'Klasyfikacja: {Xc_tr.shape[0]} próbek treningowych, {Xc_tr.shape[1]} cech')
print(f'Regresja:     {Xh_tr.shape[0]} próbek treningowych, {Xh_tr.shape[1]} cech')

Klasyfikacja: 455 próbek treningowych, 30 cech
Regresja:     16512 próbek treningowych, 8 cech


## 4.2 Regresja liniowa — przewidujemy liczbę

Najprostszy model regresji. Szuka liniowej zależności między cechami a wartością. Metryka **R²** mówi ile % zmienności model wyjaśnia (1.0 = idealnie, 0.0 = bezużyteczny).

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

model = LinearRegression()
model.fit(Xh_tr, yh_tr)            # KROK 1: trening
y_pred = model.predict(Xh_te)      # KROK 2: predykcja na nowych danych

# KROK 3: ocena
print(f'R²: {r2_score(yh_te, y_pred):.4f}')
print('Model wyjaśnia tyle % zmienności cen mieszkań')

R²: 0.5758
Model wyjaśnia tyle % zmienności cen mieszkań


## 4.3 Regresja logistyczna — przewidujemy klasę

Mimo nazwy 'regresja' to klasyfikator. Zwraca prawdopodobieństwo przynależności do klasy. Zwróćcie uwagę: **ten sam interfejs** `fit()` / `predict()` co regresja liniowa.

In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model = LogisticRegression(random_state=42)
model.fit(Xc_tr_s, yc_tr)          # ten sam fit()
y_pred = model.predict(Xc_te_s)    # ten sam predict()

# accuracy = % poprawnych predykcji
print(f'Accuracy: {accuracy_score(yc_te, y_pred):.4f}')

Accuracy: 0.9737


## 4.4 Drzewo decyzyjne i overfitting

Drzewo zadaje serię pytań tak/nie. Im głębsze, tym bardziej skomplikowane reguły. **Overfitting**: drzewo bez limitu zapamiętuje dane treningowe (train=1.0) ale słabo radzi sobie z nowymi (test spada).

In [17]:
from sklearn.tree import DecisionTreeClassifier

print(f"{'max_depth':>10} {'train':>8} {'test':>8}  komentarz")
print('-' * 45)

for depth in [3, 5, None]:
    m = DecisionTreeClassifier(max_depth=depth, random_state=42)
    m.fit(Xc_tr, yc_tr)
    tr = accuracy_score(yc_tr, m.predict(Xc_tr))   # accuracy na treningu
    te = accuracy_score(yc_te, m.predict(Xc_te))   # accuracy na teście

    if depth is None:
        komentarz = '← overfitting! train=1.0, test niższy'
    else:
        komentarz = ''
    print(f'{str(depth):>10} {tr:>8.4f} {te:>8.4f}  {komentarz}')

# Drzewo bez limitu (None) zapamiętało każdą próbkę treningową —
# ale to nie znaczy że dobrze generalizuje na nowe dane.

 max_depth    train     test  komentarz
---------------------------------------------
         3   0.9780   0.9474  
         5   0.9956   0.9474  
      None   1.0000   0.9474  ← overfitting! train=1.0, test niższy


## 4.5 Random Forest — wiele drzew razem

Zamiast jednego drzewa budujemy las (domyślnie 100 drzew). Każde trenuje na losowej próbce danych i cech, wynik to głosowanie. Błędy poszczególnych drzew się uśredniają — mniej overfittingu przy lepszych wynikach.

In [18]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,    # 100 drzew w lesie
    random_state=42
)
rf.fit(Xc_tr, yc_tr)     # ten sam interfejs fit/predict

tr = accuracy_score(yc_tr, rf.predict(Xc_tr))
te = accuracy_score(yc_te, rf.predict(Xc_te))
print(f'train: {tr:.4f}')
print(f'test:  {te:.4f}')

# train=1.0 jak drzewo bez limitu, ALE test wyższy.
# 100 drzew które każde overfittuje, razem overfittują mniej.

train: 1.0000
test:  0.9649


## 4.6 Ewaluacja — metryki klasyfikacji

Accuracy to za mało. **Confusion matrix** pokazuje typy błędów. **Precision/Recall** rozróżniają je. **AUC** ocenia jakość niezależnie od progu.

W diagnozie raka kluczowy jest recall dla klasy złośliwej — nie chcemy przeoczyć chorego.

In [19]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = rf.predict(Xc_te)
y_prob = rf.predict_proba(Xc_te)[:, 1]    # prawdopodobieństwa, nie tylko klasy

# Confusion matrix: wiersze=rzeczywiste, kolumny=przewidziane
print('Confusion matrix:')
print(confusion_matrix(yc_te, y_pred))
print()

# Pełny raport: precision, recall, f1 dla każdej klasy
print(classification_report(yc_te, y_pred, target_names=['złośliwy', 'łagodny']))

# AUC: 1.0=idealny, 0.5=losowy
print(f'AUC: {roc_auc_score(yc_te, y_prob):.4f}')

Confusion matrix:
[[40  3]
 [ 1 70]]

              precision    recall  f1-score   support

    złośliwy       0.98      0.93      0.95        43
     łagodny       0.96      0.99      0.97        71

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114

AUC: 0.9953


## 4.7 Cross-validation — wiarygodna ocena

Pojedynczy podział train/test zależy od losu. **Cross-validation** dzieli dane na k części, trenuje k razy (każda część raz jest testem), uśrednia wyniki. Odchylenie standardowe mówi jak stabilny jest model.

In [20]:
from sklearn.model_selection import cross_val_score

# cv=5 → 5-krotna walidacja krzyżowa
scores = cross_val_score(rf, Xc_tr, yc_tr, cv=5, scoring='accuracy')

print(f'Wyniki z 5 foldów: {scores.round(4)}')
print(f'Średnia:           {scores.mean():.4f}')
print(f'Odch. standardowe: {scores.std():.4f}  (niskie = model stabilny)')

Wyniki z 5 foldów: [0.978  0.9451 0.978  0.956  0.9341]
Średnia:           0.9582
Odch. standardowe: 0.0176  (niskie = model stabilny)


## 4.8 Porównanie wszystkich modeli

Zestawienie wszystkiego co przerobiliśmy w bloku ML. Kolumna **gap** (train − test) pokazuje overfitting — im mniejsza, tym lepsza generalizacja.

In [21]:
from sklearn.tree import DecisionTreeClassifier

modele = {
    'Regresja logistyczna':  (LogisticRegression(random_state=42), Xc_tr_s, Xc_te_s),
    'Drzewo (depth=5)':      (DecisionTreeClassifier(max_depth=5, random_state=42), Xc_tr, Xc_te),
    'Drzewo (bez limitu)':   (DecisionTreeClassifier(random_state=42), Xc_tr, Xc_te),
    'Random Forest':         (RandomForestClassifier(n_estimators=100, random_state=42), Xc_tr, Xc_te),
}

print(f"{'Model':25s} {'train':>8} {'test':>8} {'gap':>8}")
print('-' * 53)
for nazwa, (model, Xtr, Xte) in modele.items():
    model.fit(Xtr, yc_tr)
    tr = accuracy_score(yc_tr, model.predict(Xtr))
    te = accuracy_score(yc_te, model.predict(Xte))
    print(f'{nazwa:25s} {tr:>8.4f} {te:>8.4f} {tr-te:>8.4f}')

# Random Forest: wysokie test accuracy + mały gap = najlepszy balans

Model                        train     test      gap
-----------------------------------------------------
Regresja logistyczna        0.9868   0.9737   0.0131
Drzewo (depth=5)            0.9956   0.9474   0.0482
Drzewo (bez limitu)         1.0000   0.9474   0.0526
Random Forest               1.0000   0.9649   0.0351


# Podsumowanie


| Blok | Czego się nauczyliśmy |
|---|---|
| **OOP** | klasy, dziedziczenie, dataclasses, ABC |
| **Pydantic** | walidacja danych wchodzących do programu |
| **NLP** | tekst jako dane: bag of words → spaCy → transformery |
| **ML** | regresja, drzewa, random forest, ewaluacja |

## Główny wzorzec

Każdy model w sklearn ma **ten sam interfejs**:

```python
model = JakikolwiekModel()      # stwórz
model.fit(X_train, y_train)     # naucz
model.predict(X_test)           # przewiduj
```

To jest dokładnie wzorzec **klasy abstrakcyjnej (ABC)** który przerabialiśmy na początku semestru — klasa bazowa definiuje `fit()` i `predict()`, a konkretne modele je implementują.

**Od `class Animal` do `RandomForestClassifier` — ten sam fundament